# Multi-Modal ALS Feature Extraction Pipeline

## Architectural Overview
This notebook acts as the **Feature Engineering** stage for the downstream Vision Transformer (ViT).

**The Problem:** Raw 3D MRI scans are too large for standard Transformers.
**The Solution:** We use 3D CNNs to compress these scans into dense, meaningful **Feature Vectors**.

## Workflow
1.  **Supervised Pre-training:** We temporarily attach a classification head to the CNNs and train them on the ALS labels. This forces the CNNs to learn features that differentiate ALS from Controls. (Loss: `BCEWithLogitsLoss`).
2.  **Visualization (QC):** We use Grad-CAM to verify the CNNs are looking at the brain, not the background. Includes a masking fix for clear 3D heatmaps.
3.  **Feature Extraction:** We discard the classification head. We pass the dataset through the trained CNNs and save the output vectors (`T1_vector`, `T2_vector`, `FLAIR_vector`) for the future ViT.

In [ ]:
import os
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import torch.nn.functional as F
import nibabel as nib
from tqdm.notebook import tqdm
from sklearn.model_selection import StratifiedShuffleSplit

try:
    from dataset import MultiModalALSDataset
    from classifier import ALSTriStreamClassifier
    print("Modules imported successfully.")
except ImportError as e:
    print(f"Error importing local modules: {e}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# --- Configuration ---
DATA_DIR = "../../Data/processed"
OUTPUT_DIR = "../cnn_features"
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "encoder_weights.pth")
FEATURES_SAVE_DIR = os.path.join(OUTPUT_DIR, "extracted_features")

BATCH_SIZE = 2
LR = 1e-3
EPOCHS = 30
WARMUP_EPOCHS = 3
FORCE_RETRAIN = False

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FEATURES_SAVE_DIR, exist_ok=True)

In [ ]:
if os.path.exists(DATA_DIR):
    full_dataset = MultiModalALSDataset(rootDirectory=DATA_DIR, transform=True)
    print(f"Dataset found: {len(full_dataset)} samples")
    
    if len(full_dataset) > 0:
        # Extract labels for stratified splitting
        labels = np.array([s['label'] for s in full_dataset.samples])
        n_pos = int(labels.sum())
        n_neg = len(labels) - n_pos
        print(f"Class distribution: {n_neg} controls, {n_pos} ALS patients")
        
        # Stratified split: keeps class ratios equal in train and val
        splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
        train_idx, val_idx = next(splitter.split(np.zeros(len(labels)), labels))
        
        train_set = Subset(full_dataset, train_idx)
        val_set = Subset(full_dataset, val_idx)
        
        print(f"Train: {len(train_set)} | Val: {len(val_set)}")
        
        # Verify stratification
        train_labels = labels[train_idx]
        val_labels = labels[val_idx]
        print(f"Train ALS ratio: {train_labels.mean():.2f} | Val ALS ratio: {val_labels.mean():.2f}")
        
        train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)
        full_loader = DataLoader(full_dataset, batch_size=1, shuffle=False)
        
        # Compute pos_weight for imbalanced classes
        pos_weight = torch.tensor([n_neg / n_pos]).to(DEVICE)
        print(f"pos_weight: {pos_weight.item():.2f}")
else:
    print(f"Error: Data directory {DATA_DIR} not found.")

## Supervised Pre-training


In [ ]:
model = ALSTriStreamClassifier().to(DEVICE)

# Weighted loss to handle class imbalance
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)

# Cosine annealing schedule (applied after warmup)
cosine_scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS - WARMUP_EPOCHS, eta_min=1e-6
)

# Linear warmup: ramp LR from ~0 to LR over WARMUP_EPOCHS
warmup_scheduler = optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_EPOCHS
)

# Combine: warmup first, then cosine
scheduler = optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[WARMUP_EPOCHS]
)

# Mixed Precision
use_amp = (DEVICE.type == 'cuda')
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

if os.path.exists(CHECKPOINT_PATH) and not FORCE_RETRAIN:
    print(f"Loading existing weights from {CHECKPOINT_PATH}...")
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
else:
    print(f"Starting Supervised Pre-training (AMP Enabled: {use_amp})...")
    best_val_loss = float('inf')
    patience_counter = 0
    EARLY_STOP_PATIENCE = 8
    
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
        
        for (t1, t2, flair), label in loop:
            t1, t2, flair = t1.to(DEVICE), t2.to(DEVICE), flair.to(DEVICE)
            label = label.to(DEVICE).unsqueeze(1)
            
            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast(enabled=use_amp):
                output = model(t1, t2, flair)
                loss = criterion(output, label)
            
            scaler.scale(loss).backward()
            # Gradient clipping to stabilize training
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item()
            
            with torch.no_grad():
                preds = (torch.sigmoid(output) > 0.5).float()
                correct += (preds == label).sum().item()
                total += label.size(0)
            
            loop.set_postfix(loss=loss.item())
        
        train_loss = running_loss / len(train_loader)
        train_acc = correct / total
        
        #Validation Phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for (t1, t2, flair), label in val_loader:
                t1, t2, flair = t1.to(DEVICE), t2.to(DEVICE), flair.to(DEVICE)
                label = label.to(DEVICE).unsqueeze(1)
                
                with torch.cuda.amp.autocast(enabled=use_amp):
                    output = model(t1, t2, flair)
                    loss = criterion(output, label)
                
                val_loss += loss.item()
                preds = (torch.sigmoid(output) > 0.5).float()
                val_correct += (preds == label).sum().item()
                val_total += label.size(0)
        
        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        
        # Step the LR scheduler (warmup then cosine)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        
        # Track history
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        
        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f} Acc={train_acc:.2f} | Val Loss={val_loss:.4f} Acc={val_acc:.2f} | LR={current_lr:.6f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), CHECKPOINT_PATH)
            print("-> Saved Best Model")
        else:
            patience_counter += 1
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"Early stopping at epoch {epoch+1} (no improvement for {EARLY_STOP_PATIENCE} epochs)")
                break
    
    # Load best weights after training
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
    print("Loaded best model weights.")

In [ ]:
# Plot training curves
if 'train_losses' in locals() and len(train_losses) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(train_losses, label='Train Loss')
    ax1.plot(val_losses, label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Loss Curves')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(train_accs, label='Train Acc')
    ax2.plot(val_accs, label='Val Acc')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Accuracy Curves')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Phase 2: Visualization (Grad-CAM)
Verify that the CNNs are focusing on anatomy, not noise.

In [ ]:
class GradCAM:
    def __init__(self, model_part, target_layer):
        self.model_part = model_part
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.handle_fwd = self.target_layer.register_forward_hook(self.save_activation)
        self.handle_bwd = self.target_layer.register_full_backward_hook(self.save_gradient)
    def save_activation(self, module, input, output): self.activations = output
    def save_gradient(self, module, grad_input, grad_output): self.gradients = grad_output[0]
    def remove_hooks(self): self.handle_fwd.remove(); self.handle_bwd.remove()
    def generate(self, x):
        output = self.model_part(x)
        self.model_part.zero_grad()
        output.mean().backward()
        weights = torch.mean(self.gradients, dim=[0, 2, 3, 4])
        cam = torch.zeros(self.activations.shape[2:], dtype=torch.float32).to(DEVICE)
        for i, w in enumerate(weights): cam += w * self.activations[0, i, :, :, :]
        cam = F.relu(cam)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam = cam.unsqueeze(0).unsqueeze(0)
        cam = F.interpolate(cam, size=x.shape[2:], mode='trilinear', align_corners=False)
        mask = (x > 0.05).float()
        cam = cam * mask
        return cam.squeeze().detach().cpu().numpy()

def run_qc(index=0):
    model.eval()
    if len(full_dataset) <= index: print("Index out of bounds"); return
    (t1, t2, flair), label = full_dataset[index]
    t1_in = t1.unsqueeze(0).to(DEVICE).requires_grad_(True)
    t2_in = t2.unsqueeze(0).to(DEVICE).requires_grad_(True)
    fl_in = flair.unsqueeze(0).to(DEVICE).requires_grad_(True)
    enc_t1, enc_t2, enc_fl = model.model.t1Encoder, model.model.t2Encoder, model.model.flairEncoder
    g_t1 = GradCAM(enc_t1, enc_t1.layer2)
    g_t2 = GradCAM(enc_t2, enc_t2.layer2)
    g_fl = GradCAM(enc_fl, enc_fl.layer2)
    m_t1 = g_t1.generate(t1_in)
    m_t2 = g_t2.generate(t2_in)
    m_fl = g_fl.generate(fl_in)
    s = m_t1.shape[0] // 2
    plt.figure(figsize=(15, 5))
    plt.subplot(1, 3, 1); plt.imshow(t1.numpy()[0, s], cmap='gray'); plt.imshow(m_t1[s], cmap='jet', alpha=0.5); plt.title("T1")
    plt.subplot(1, 3, 2); plt.imshow(t2.numpy()[0, s], cmap='gray'); plt.imshow(m_t2[s], cmap='jet', alpha=0.5); plt.title("T2")
    plt.subplot(1, 3, 3); plt.imshow(flair.numpy()[0, s], cmap='gray'); plt.imshow(m_fl[s], cmap='jet', alpha=0.5); plt.title("FLAIR")
    plt.suptitle(f"Subject: {full_dataset.samples[index]['id']} | Label: {'ALS' if label.item() == 1.0 else 'Control'}")
    plt.show()
    g_t1.remove_hooks(); g_t2.remove_hooks(); g_fl.remove_hooks()

run_qc(0)

## Phase 3: Feature Extraction
Generate and save pure feature vectors for the ViT.

In [ ]:
# Disable augmentation for deterministic feature extraction
full_dataset.transform = False

def extract():
    model.eval()
    with torch.no_grad():
        for i, ((t1, t2, flair), label) in enumerate(tqdm(full_loader, desc="Extracting")):
            t1, t2, flair = t1.to(DEVICE), t2.to(DEVICE), flair.to(DEVICE)
            f1 = model.model.t1Encoder(t1).cpu()
            f2 = model.model.t2Encoder(t2).cpu()
            ff = model.model.flairEncoder(flair).cpu()
            sid = full_loader.dataset.samples[i]['id']
            torch.save({'id': sid, 'label': label.item(), 't1': f1, 't2': f2, 'fl': ff}, os.path.join(FEATURES_SAVE_DIR, f"{sid}_features.pt"))
    print("Done.")

extract()